In [ ]:
%pylab inline
import svgpathtools as spt
import eucare as ec
import networkx as nx

In [ ]:
def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

#max_kawasaki_sum(G)

In [ ]:
"""
Plan for cleaner and more efficient overlap-graph creation:

0. Start with list of N line segments, as array of shape (N, 2, 2)
1. Find identical segments (reshape to (N, 4) , cluster with group_closeby())
2. Find crossings of unique segments with sweeping line algorithm
3. Group close crossings
4. Find segments very close to crossings; Add these to the crossings (This step is new!!)
5. Construct nx graph

7. return: the nx graph, a mapping from the indices of original segments to the edges in the graph

To use this for cleaning of e.g. svg-imported CPs, provide addtional method that 'cleans' the CP: 
Join all straight degree 2 vertices that 

"""

from eucare.overlap import group_closeby, get_potential_intersections, line_segment_intersections
from eucare.utils import VerboseTimer
from eucare.conversions import EHEG_from_nx
from tqdm.auto import tqdm

def get_first_occurences(clustering):
    return np.argmax(clustering[None] == np.arange(np.max(clustering) + 1)[:, None], axis=1)

def segments_to_clean_graph(line_segments, eps):
    timer = VerboseTimer()
    line_segments = np.array(line_segments, dtype=np.float32)
    assert line_segments.shape[1:] == (2, 2), f'{line_segments.shape}'
    initial_n_segments = len(line_segments)
    
    # 1. Find identical segments (reshape to (N, 4) , cluster with group_closeby())
    line_segment_clustering = group_closeby(line_segments.reshape(-1, 4), eps)
    line_segments = line_segments[get_first_occurences(line_segment_clustering)]
    print(line_segments.shape)
    print(np.max(line_segment_clustering))
    timer.round()
    
    # 2. Get Itersections
    potential_intersections = get_potential_intersections(line_segments)
    print(len(potential_intersections))
    timer.round()
    
    crossings = []
    crossings_to_edges = []
    for i, j in potential_intersections:
        l1, l2 = line_segments[i], line_segments[j]
        intersections = line_segment_intersections(l1, l2, eps=eps)
        if not intersections:
            continue
        crossings.extend(intersections)
        for _ in range(len(intersections)):
            crossings_to_edges.append((i, j))
    crossings = np.array(crossings, dtype=np.float32)
    
    # add extra crossings at ends of segments
    crossings = np.concatenate([crossings, line_segments[:, 0], line_segments[:, 1]], axis=0)
    crossings_to_edges.extend(np.arange(len(line_segments))[:, None])
    crossings_to_edges.extend(np.arange(len(line_segments))[:, None])
    
    timer.round(f'{len(crossings)} crossings')
    
    # 3. Group close crossings
    crossing_clustering = group_closeby(crossings, eps=eps)
    crossings = crossings[get_first_occurences(crossing_clustering)]
    
    edges_to_crossings = [set() for i in range(len(line_segments))]
    for i, edge_ids in enumerate(crossings_to_edges):
        for e in edge_ids:
            edges_to_crossings[e].add(crossing_clustering[i])
    
    timer.round(f'crossings clustered, now {len(crossings)}')
    
    # 4. Find segments very close to crossings; Add these to the crossings (This step is new!!)
    crossings_and_segments = np.concatenate([crossings[:, None, :].repeat(2, axis=1), line_segments], axis=0)
    print(crossings_and_segments.shape)
    potential_intersections = get_potential_intersections(crossings_and_segments)
    print(len(potential_intersections))
    for (i, j) in potential_intersections:
        if i > j:
            i, j = j, i
        if i >= len(crossings) or j < len(crossings):
            continue
        l1, l2 = crossings_and_segments[i], crossings_and_segments[j]
        intersections = line_segment_intersections(l1, l2, eps=eps)
        if intersections:
            edges_to_crossings[j-len(crossings)].add(i)
    
    # 5. Construct nx graph
    nx_edges = []
    edge_to_ordered_ids = []
    for i in tqdm(range(len(line_segments))):
        e = line_segments[i]
        crossing_ids = np.array(list(edges_to_crossings[i]), dtype=np.int32)
        crossing_positions = crossings[crossing_ids]
        progression_along_edge = ((crossing_positions - e[0][None]) * (e[1] - e[0])[None]).sum(-1)
        order = np.argsort(progression_along_edge)
        ordered_ids = crossing_ids[order]
        edge_to_ordered_ids.append(ordered_ids)
        nx_edges.append(np.stack([ordered_ids[:-1], ordered_ids[1:]], axis=-1))
    nx_edges = np.concatenate(nx_edges)
    timer.round('crossing orders')

    print('constructing nx graph..')
    nx_graph = nx.Graph()
    nx_graph.add_edges_from(nx_edges)
    nx_positions = {i: pos for i, pos in enumerate(crossings)}
    timer.round('nx graph')
    print('order of nx graph:', nx_graph.order())
    print('converting to EHEG..')

    overlap_G, v_lookup = EHEG_from_nx(nx_graph, nx_positions, return_v_lookup=True)
    overlap_G.recompute_lengths_and_angles()
    timer.round('EHEG')
    
    return overlap_G

In [ ]:
render_settings = dict(
    width=1500, height=1500,
    figsize=(7, 7),
    scale='auto',
    render_edges=True,
    render_faces=False,
    render_vertices=False,
    line_width=5,
    face_inset=0.000,
    for_cutting=False
)

render_settings_for_cutting = dict(
    width=1500, height=1500,
    figsize=(7, 7),
    scale='auto',
    render_edges=True,
    render_faces=False,
    render_vertices=False,
    line_width=5,
    face_inset=0.000,
    for_cutting=True
)

In [ ]:
import os
#print(os.listdir('/home/roman/Documents/Origami/RobertLangCPs/'))

#filepath = '/home/roman/Documents/Origami/Oripa/tato.svg'
#filepath = '/home/roman/Documents/Origami/Oripa/minotaur.svg'
#filepath = '/home/roman/Documents/Origami/RobertLangCPs/classical_cicada_cp.svg'
#filepath = '/home/roman/Downloads/randlettflappingbird.svg'

# filepath = 'nice_images/verrillO_7/CP.svg'
# filepath = 'nice_images/hyperbolic_7-3_join_small/CP.svg'
filepath = '/home/roman/Downloads/test1_twists_scaled_printReady.svg'
paths, attributes = spt.svg2paths(filepath)

In [ ]:
from eucare.overlap import MOUNTAIN, VALLEY, CREASE_ASSIGNMENT
import re
from collections import defaultdict
import operator
# step 1: 

points = []
edges = []

def get_stroke(attrs):
    hits = re.search('stroke:(#.{6})', attrs['style'])
    if hits is not None:
        hits = hits.groups()
    if not hits:
        return None
    elif len(hits) == 1:
        return hits[0]
    else:
        assert False, f'Found multiple strokes: {list(hits)}'
    
counts_by_stroke = defaultdict(int)
counts_by_style = defaultdict(int)

for path, attrs in zip(paths, attributes):
    #if len(path) > 1:
    #    continue
    for line in path:
        assert isinstance(line, spt.path.Line)
        #line = path[0]
        start = np.array([line.start.real, line.start.imag], dtype=np.float32)
        end = np.array([line.end.real, line.end.imag], dtype=np.float32)
        #print(start, end)
        crease_type = None
        # this works for cps exported from oripa
        if 'style' in attrs:
            if 'red' in attrs['style']:
                crease_type = MOUNTAIN
            elif 'blue' in attrs['style']:
                crease_type = VALLEY
            elif 'gray' in attrs['style']:
                continue
        edge_attrs = dict() if crease_type is None else {CREASE_ASSIGNMENT: crease_type}
        
        # this is e.g. for robert langs cps
        if len(path) == 1: 
            if 'style' in attrs:
                counts_by_stroke[get_stroke(attrs)] += 1
                counts_by_style[attrs['style']] += 1
                stroke = get_stroke(attrs)
                edge_attrs['svg_stroke'] = stroke 
            elif 'stroke' in attrs:
                counts_by_stroke[attrs['stroke']] += 1
                edge_attrs['svg_stroke'] = attrs['stroke'] 
            
        edges.append((len(points), len(points)+1, edge_attrs))
        points.extend([start, end])
points = np.stack(points)
points -= np.mean(points, axis=0)
points /= 2*np.max(np.abs(points))
points += [[0.5, 0.5]]

clustering = ec.overlap.group_closeby(points, 1e-5)
#print(np.concatenate([points, clustering[:, None]], axis=1))
first_occurences = np.argmax(clustering[None] == np.arange(np.max(clustering) + 1)[:, None], axis=1)
merged_points = points[first_occurences]

line_segments = np.array([[merged_points[clustering[i]], merged_points[clustering[j]]] 
                         for i, j, attrs in edges])

# line_segments = np.array([[points[i], points[j]] 
#                           for i, j, attrs in edges])
print(line_segments.shape)

from eucare.plotting import plot_lines
plot_lines(line_segments)

#G = segments_to_clean_graph(line_segments, eps=1e-3)

G = nx.Graph()
G.add_edges_from([(tuple(merged_points[clustering[i]]), tuple(merged_points[clustering[j]]), attrs) 
                  for i, j, attrs in edges])

G = ec.conversions.EHEG_from_nx(G)



if len(counts_by_stroke) not in (2, 3):
    pass
else:
    if len(counts_by_stroke) == 3:  # assume one is the border stroke
        on_border_counts = {key: 0 for key in list(counts_by_stroke.keys()) + [None]}
        for e in G.border_edges():
            on_border_counts[e.attributes.get('svg_stroke', None)] += 1
            on_border_counts[e.rev.attributes.get('svg_stroke', None)] += 1
        del on_border_counts[None]
        border_stroke = max(on_border_counts.items(), key=operator.itemgetter(1))[0]
        del on_border_counts[border_stroke]
        crease_strokes = sorted(on_border_counts)
    else: # 2 strokes
        crease_strokes = sorted(counts_by_stroke)
    for e in G.halfedges:
        stroke = e.attributes.get('svg_stroke', None)
        if stroke in crease_strokes:
            e[CREASE_ASSIGNMENT] = MOUNTAIN if stroke == crease_strokes[0] else VALLEY


# G.normalize_positions()
# G.show(**render_settings)

G.show()
#print(counts_by_stroke)
print(counts_by_stroke.values())

In [ ]:
from eucare.overlap import save_results, fold_complete
G.recompute_lengths_and_angles()
result = fold_complete(G.copy(), overlap_eps=1e-6, area_eps=0)
render_settings['render_faces'] = False

In [ ]:
import os
from eucare.redering import SvgwriteRenderer
from eucare.overlap import save_results

path = 'nice_images/hyperbolic_7-3_join_small_rerendered'

bbox = (25, 20)
# bbox = (5, 20)
# bbox = (35, 30)
# bbox = (65, 48)
# bbox = (95, 58)

render_settings = dict(line_width=0.003, face_inset=0, render_vertices=False, render_faces=False, height=2048)
save_results(result, path, bbox=bbox, render_settings=render_settings)

In [ ]:
c = 0
for v in G.vertices:
    if not v.on_border() and v.order() % 2 == 1:
        c += 1
c

In [ ]:
ec.overlap.fold_wireframe(G)
render_settings['render_faces'] = False
G.show(**render_settings)
ec.overlap.fold_wireframe(G)

In [ ]:
#cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area()//0.0001)())
cc = ec.classifiers.congruency_classifier()
for f in G.faces:
    f['color_key'] = cc.classify(f)
ec.overlap.color_creases(G)
render_settings['render_faces'] = False
G.show(**render_settings)
max_kawasaki_sum(G)

#SRG = ec.reciprocal_figures.shrink_rotate_graph(G)
#SRG.recompute_lengths_and_angles()
#SRG.show(**render_settings)

In [ ]:
from eucare.redering import SvgwriteRenderer

#G.show(filename='resave', **render_settings_for_cutting)
renderer = SvgwriteRenderer()
renderer.render_graph('nice_images/verillO7.svg', G, height=20)

In [ ]:
from eucare.overlap import fold_complete
result = fold_complete(G.copy(), overlap_eps=1e-5, area_eps=0)
result['folded_view_top'].show(**render_settings)
result['folded_view_bottom'].show(**render_settings)
result['CP'].show(**render_settings)

In [ ]:
initial_face = None
for f in G.faces:
    pos = np.array([v['pos'] for v in f.vertex_iter()])
    if np.all(np.min(pos, axis=0) <= 0) and np.all(np.max(pos, axis=0) > 0):
        initial_face = f
G.twocolor_faces(initial_face=initial_face)
G.show(**render_settings)
for f in filter(lambda f: f['color_key'], G.faces):
    for e in f.halfedge_iter():
        e['in_angle'] *= -1
G.recompute_positions()
print(max_kawasaki_sum(G))
G.show(**render_settings)

In [ ]:
def get_over_under_pairs(G, two_coloring_key='color_key'):
    # return list of pairs (f1, f2) with f1 over f2
    # G is assumed to be two-colored
    over_under_pairs = []
    for e in G.halfedges:
        crease_type = e.attributes.get('crease_type', None)
        if crease_type in ('mountain', 'valley') and not (e.on_border() or e.rev.on_border()):
            e_above = e if e.face[two_coloring_key] else e.rev
            if crease_type is 'mountain':
                e_above = e_above.rev
            over_under_pairs.append([e_above.face, e_above.rev.face])
    print('number of pairs', len(over_under_pairs))
    return over_under_pairs

over_under_pairs = get_over_under_pairs(G)
G_over = ec.overlap.overlap_graph(G, eps=1e-4)

In [ ]:
G_over.show(**render_settings)

In [ ]:
from collections import defaultdict
area_threshold = 1e-6

cc = ec.classifiers.lambda_classifier(lambda f: f.area() > area_threshold)()
counts = defaultdict(int)
for f in G_over.faces:
    # over = 0
    # under = 0
    # for e in f.halfedge_iter():
    #     print(e.attributes)
    #f['color_key'] = over / (over + under)
    f['color_key'] = cc.classify(f)
    counts[f['color_key']] += 1
print(counts)
print()
G_over.show(**render_settings)

In [ ]:
#central_face = next(iter(f for f in G.faces if f.order() == ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
#    G = ec.half.EuclideanPositionHEG(other=G)24))
#over_under_pairs = [(e.rev.face, e.rev.nex.rev.face) for e in central_face.halfedge_iter()]

In [ ]:
ec.overlap.find_folded_face_order(G_over, [], ignore_area_threshold=1e-6)  #, over_under_pairs)

#for e in G.halfedges:
#    print(e['original_face_groups'])


TOP = 'top_side'
BOTTOM = 'bottom_side'


def show_folded(G, side=TOP):
    assert side in (TOP, BOTTOM)
    cc = ec.classifiers.CountingClassifier(ec.classifiers.RepresentationClassifier())
    G = G.copy()
    for f in G.faces:
        #f['color_key'] = len(f['original_faces'])
        try:
            f['color_key'] = cc.classify(f['sorted_original_faces'][0 if side is TOP else -1])
        except IndexError:
            f['color_key'] = 1000
        #print(f['color_key'])
    G.show(**render_settings)

    to_delete = [e
                 for e in G.halfedges
                 if not (e.on_border() or e.rev.on_border()) and e.face['color_key'] is e.rev.face['color_key']]

    G.halfedges.difference_update(to_delete)
    G = ec.conversions.EHEG_from_nx(G.to_networkx_undirected(), {v: v['pos'] for v in G.vertices})
#     to_join = []
#     for v in G.vertices:
#         if not v.on_border() and v.order() == 2:
#             to_join.append(v)
#     for v in to_join:
#         G.join_vertex(v)
    G.recompute_lengths_and_angles()
    cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area()//0.0001)())
    for f in G.faces:
        # over = 0
        # under = 0
        # for e in f.halfedge_iter():
        #     print(e.attributes)
        #f['color_key'] = over / (over + under)
        f['color_key'] = cc.classify(f)
    G.show(**render_settings)

show_folded(G_over, BOTTOM)
show_folded(G_over, TOP)

In [ ]:
#to solve this properly: every triplet gets an area; 

In [ ]:
rho = {frozenset({(16, 17), (43, 45)}), frozenset({(50, 31), (47, 29), (18, 19)}), frozenset({(11, 9), (0, 10)}), frozenset({(6, 42), (43, 45)}), frozenset({(20, 21), (52, 34), (33, 54)}), frozenset({(32, 30), (51, 42), (13, 58), (14, 22)}), frozenset({(51, 42), (55, 45), (1, 7), (14, 22)}), frozenset({(28, 17), (43, 27)}), frozenset({(47, 54), (53, 15), (33, 29), (52, 50), (34, 31), (25, 26), (20, 18), (21, 19)}), frozenset({(8, 12), (48, 44)}), frozenset({(36, 37), (40, 38), (41, 39)}), frozenset({(43, 2), (6, 5)}), frozenset({(30, 0), (11, 13)}), frozenset({(32, 51), (58, 22), (30, 13)}), frozenset({(9, 10)}), frozenset({(1, 56), (57, 55)}), frozenset({(18, 22), (19, 58), (13, 47), (30, 29), (50, 51), (14, 15), (32, 31), (42, 26)}), frozenset({(51, 5), (6, 42), (55, 2), (43, 45)}), frozenset({(13, 58), (57, 0), (1, 7), (11, 56), (32, 30), (51, 42), (55, 45), (14, 22)}), frozenset({(48, 49), (12, 3)}), frozenset({(39, 3), (23, 24), (46, 37), (16, 35), (36, 49), (40, 10), (41, 4), (9, 38)}), frozenset({(1, 7), (14, 22)}), frozenset({(18, 15), (19, 47), (31, 29), (50, 26)}), frozenset({(44, 45), (16, 46), (8, 7), (4, 24)}), frozenset({(8, 7), (4, 24)}), frozenset({(30, 13)}), frozenset({(46, 44), (8, 4)}), frozenset({(49, 10), (16, 46), (9, 3), (4, 24)}), frozenset({(55, 51), (1, 22)}), frozenset({(20, 53), (52, 25), (21, 54), (34, 33)}), frozenset({(11, 7), (0, 45)}), frozenset({(44, 45), (49, 10), (9, 3), (48, 0), (8, 7), (4, 24), (16, 46), (11, 12)}), frozenset({(58, 56), (57, 32)}), frozenset({(16, 17), (46, 28), (27, 44), (43, 45)}), frozenset({(36, 40), (39, 38), (41, 23), (37, 35)}), frozenset({(9, 10), (46, 49), (4, 3)})}

In [ ]:
counts = defaultdict(int)
for groups in rho:
    counts[len(groups)] += 1
counts

In [ ]:
import numpy as np
import eucare as ec
from eucare.example_tilesets import platonic, square_strip
from eucare.example_graphs import from_tiles
from test.tresst_overlap import render_settings

G = from_tiles(square_strip(), 5, vertex_based=False)
#G.normalize_positions()
#G.recompute_lengths_and_angles()
G.show(**render_settings)

# from eucare.overlap import fold_complete
# result = fold_complete(G.copy(), overlap_eps=1e-4, area_eps=1e-6)
# result['folded_view_top'].show(**render_settings)
# result['CP'].show(**render_settings)

ec.overlap.fold_wireframe(G)
G_over = ec.overlap.overlap_graph(G, eps=1e-6)
crease_assignment = ec.overlap.find_folded_face_order(G_over, [])  #, over_under_pairs)

ec.overlap.fold_wireframe(G)
for e in G.halfedges:
    e['crease_assignment'] = crease_assignment.get(e, 0)
ec.overlap.color_creases(G)
render_settings['render_faces'] = False
render_settings['line_width'] = 20
G.show(**render_settings)

In [ ]:

comparison_func

In [ ]:
list(G_over.faces)[0].attributes

In [ ]:
s = 0
for e in G_over.halfedges:
    s += len(e['original_edges'])
    print(len(e['original_edges']), e)
    for e2 in e['original_edges']:
        pass
        #print(e2)
print(s)
print(len(G.halfedges))

In [ ]:
s = 0
for e in G_over.halfedges:
    groups = e['original_face_groups']
    for g in groups:
        print(len(g))
    print()
    print(len(groups))
    print()
    s += len(e['original_face_groups'])
    
print(s)
print(len(G.halfedges))